In [ ]:
import os
from dotenv import load_dotenv
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document
from langchain_community.vectorstores import Chroma

import numpy as np
from typing import List

from langchain_openai import ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

# 1. Document Loading

In [2]:
loader = DirectoryLoader(
    path='../data/txt',
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding":"utf-8"},
    show_progress=True
)
documents = loader.load()

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 12.22it/s]


# 2. Chunking

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(documents=documents)

len(chunks)

34

# 3. Apply Embedding to the chunks and store in Vector Store

In [4]:
store_location = "../vector_store/chroma_db"
embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small')
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    persist_directory=store_location,
    collection_name="basic_rag_collection"
)

print(f"Number of vectors created: {vector_store._collection.count()}")

Number of vectors created: 68


# 4. Perform Similarity Search

In [5]:
query = """What are some necessary documents to purchase land in India?"""

In [6]:
similar_docs = vector_store.similarity_search(query=query, k=3)

In [7]:
for idx,doc in enumerate(similar_docs):
    print(f"Doc {idx+1}: {doc.page_content}")
    print("\n")
    print(f'Source: {doc.metadata.get("source", "unknown")}')

Doc 1: When buying farmland in India, you must check land title deeds and encumbrance certificates to ensure clear ownership and no financial liabilities, along with revenue records like the Record of Rights


Source: ..\data\txt\india real estate checklist.txt
Doc 2: When buying farmland in India, you must check land title deeds and encumbrance certificates to ensure clear ownership and no financial liabilities, along with revenue records like the Record of Rights


Source: ..\data\txt\india real estate checklist.txt
Doc 3: To buy farmland in Karnataka, you must gather essential documents including the seller's ID and PAN cards, Title Deed, Sale Deed, Encumbrance Certificate, Property Tax receipts, and Record of Rights


Source: ..\data\txt\karnataka document checklist.txt


# Similarity Search with Scores

In [8]:
results_with_scores = vector_store.similarity_search_with_score(query=query, k=3)

In [9]:
for idx, doc in enumerate(results_with_scores):
    print(f"Doc {idx+1}: Score: {doc[1]:.3f}")

Doc 1: Score: 0.619
Doc 2: Score: 0.619
Doc 3: Score: 0.666


# 5. Augmentation and Generation

## 5.1 Obtain Retriever from Vector Store

In [39]:
retriever = vector_store.as_retriever(
    search_kwargs={"k":3}
)

retriever.tags

['Chroma', 'OpenAIEmbeddings']

## 5.2 Create a Prompt Template in order to talk to the LLM

In [40]:
llm = ChatOpenAI(model_name="gpt-4o-mini",
                 temperature=0.1,
                 max_tokens=500)


In [23]:
system_prompt = """You are an assistant for question answering tasks. 
Use the following pieces of retrieved context to answer the question.
If you dont know the answer just say that you dont know. 
Use a maximum of three sentences and keep the answers concise.

Context: {context}"""

In [24]:
prompt = ChatPromptTemplate([
    ("system", system_prompt),
    ("human", "{input}")
])

## 5.3 Create Document Chain

In [41]:
document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

## 5.4 Create RAG Chain

In [42]:
rag_chain = create_retrieval_chain(
    retriever,
    document_chain
)

## 5.5 Execute the Chain

In [43]:
response = rag_chain.invoke(
    {"input": "what are some documents required to purchase land?"}
)

response["answer"]

"Some documents required to purchase land include the Title Deed, which proves the seller's legal ownership and the property's history, and a Conversion Certificate if the land's use has changed from agricultural to non-agricultural."

In [44]:
for idx, doc in enumerate(response["context"]):
    print(f"Retrieved Doc{idx+1}: {doc.page_content}")
    print("\n")

Retrieved Doc1: Documents to verify seller's ownership and land history:
Title Deed: Proves the seller's legal ownership and the property's history.


Retrieved Doc2: Documents to verify seller's ownership and land history:
Title Deed: Proves the seller's legal ownership and the property's history.


Retrieved Doc3: Conversion Certificate: If the land's use has been changed from agricultural to non-agricultural, this certificate is required. 
Buyer & Seller Identification:


